In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!pip install rdflib pydotplus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.9/566.9 kB 9.9 MB/s eta 0:00:00


In [7]:
# rdf_folder_to_knowledge_graphs.py
# Batch-convert all RDF Turtle files in a folder into knowledge graph images.
# For each input .ttl, outputs: <name>.dot and <name>.png in the output folder.

import io
import sys
import shutil
from pathlib import Path

import rdflib
import pydotplus
from rdflib.tools.rdf2dot import rdf2dot

# ---PYTHON DEPENDENCIES---
# pip install rdflib pydotplus
# ---SYSTEM DEPENDENCY---
# Graphviz must be installed so pydotplus can call `dot`

def ensure_graphviz_available() -> None:
    """Raise a clear error if Graphviz 'dot' executable is not found."""
    if shutil.which("dot") is None:
        raise RuntimeError(
            "GraphViz's executables not found. Install Graphviz and ensure 'dot' is on PATH.\n"
            "Colab: run `!apt-get -y install graphviz`"
        )

def save_graph_visualization_from_turtle_file(
    ttl_path: Path,
    dot_out: Path,
    png_out: Path,
    also_write_dot: bool = True,
) -> None:
    """
    Parse a Turtle (.ttl) RDF file and save a knowledge graph visualization to PNG.
    Optionally writes the intermediate DOT file for debugging/viewing.
    """
    if not ttl_path.exists():
        print(f"Error: RDF file not found at {ttl_path.resolve()}")
        return

    # Load RDF
    graph = rdflib.Graph()
    try:
        graph.parse(ttl_path.as_posix(), format="turtle")
    except Exception as e:
        print(f"[{ttl_path.name}] Failed to parse Turtle file: {e}")
        return

    print(f"[{ttl_path.name}] Parsed {len(graph)} triples.")

    # RDF -> DOT
    dot_stream = io.StringIO()
    try:
        rdf2dot(graph, dot_stream, opts={})
    except Exception as e:
        print(f"[{ttl_path.name}] Failed to convert RDF to DOT: {e}")
        return

    dot_data = dot_stream.getvalue()

    # Write DOT
    if also_write_dot:
        try:
            dot_out.write_text(dot_data, encoding="utf-8")
            print(f"[{ttl_path.name}] Wrote DOT -> {dot_out.name}")
        except Exception as e:
            print(f"[{ttl_path.name}] Failed to write DOT: {e}")

    # DOT -> PNG
    try:
        ensure_graphviz_available()
        pydot_graph = pydotplus.graph_from_dot_data(dot_data)
        if isinstance(pydot_graph, list):  # rare case
            pydot_graph = pydot_graph[0]
        pydot_graph.write_png(png_out.as_posix())
        print(f"[{ttl_path.name}] Wrote PNG -> {png_out.name}")
    except Exception as e:
        print(
            f"[{ttl_path.name}] Error while rendering PNG. "
            f"If you see 'GraphViz's executables not found', install Graphviz.\nDetails: {e}"
        )

def process_folder(
    input_dir: str,
    output_dir: str,
    pattern: str = "*.ttl",
    also_write_dot: bool = True,
) -> None:
    """
    Convert all .ttl files in input_dir to PNG knowledge graphs in output_dir.
    """
    in_path = Path(input_dir)
    out_path = Path(output_dir)

    if not in_path.exists() or not in_path.is_dir():
        print(f"Error: input folder not found or not a directory: {in_path.resolve()}")
        sys.exit(1)

    out_path.mkdir(parents=True, exist_ok=True)

    ttl_files = sorted(in_path.glob(pattern))
    if not ttl_files:
        print(f"No files matching '{pattern}' found in {in_path.resolve()}")
        return

    print(f"Found {len(ttl_files)} .ttl file(s) in {in_path.resolve()}")
    for ttl_file in ttl_files:
        stem = ttl_file.stem
        dot_out = out_path / f"{stem}.dot"
        png_out = out_path / f"{stem}.png"
        save_graph_visualization_from_turtle_file(
            ttl_file, dot_out, png_out, also_write_dot=also_write_dot
        )

if __name__ == "__main__":
    # ==== EDIT THESE FOR COLAB OR YOUR ENVIRONMENT ====
    INPUT_FOLDER = "/content/drive/MyDrive/RA/WUR/Dataset/diagram2graph/Outputs/ZeroShot_outputs"   # folder containing .ttl files
    OUTPUT_FOLDER = "/content/kg_Out_ZeroShot"  # folder to write .dot and .png
    # ===================================================

    process_folder(INPUT_FOLDER, OUTPUT_FOLDER, pattern="*.ttl", also_write_dot=True)


Found 219 .ttl file(s) in /content/drive/MyDrive/RA/WUR/Dataset/diagram2graph/Outputs/ZeroShot_outputs
[0.ttl] Parsed 62 triples.
[0.ttl] Wrote DOT -> 0.dot
[0.ttl] Wrote PNG -> 0.png
[10.ttl] Parsed 36 triples.
[10.ttl] Wrote DOT -> 10.dot
[10.ttl] Wrote PNG -> 10.png
[100.ttl] Parsed 34 triples.
[100.ttl] Wrote DOT -> 100.dot
[100.ttl] Wrote PNG -> 100.png
[107.ttl] Parsed 54 triples.
[107.ttl] Wrote DOT -> 107.dot
[107.ttl] Wrote PNG -> 107.png
[109.ttl] Parsed 64 triples.
[109.ttl] Wrote DOT -> 109.dot
[109.ttl] Wrote PNG -> 109.png
[11.ttl] Parsed 44 triples.
[11.ttl] Wrote DOT -> 11.dot
[11.ttl] Wrote PNG -> 11.png
[111.ttl] Parsed 72 triples.
[111.ttl] Wrote DOT -> 111.dot
[111.ttl] Wrote PNG -> 111.png
[113.ttl] Parsed 92 triples.
[113.ttl] Wrote DOT -> 113.dot
[113.ttl] Wrote PNG -> 113.png
[114.ttl] Parsed 82 triples.
[114.ttl] Wrote DOT -> 114.dot
[114.ttl] Wrote PNG -> 114.png
[116.ttl] Parsed 105 triples.
[116.ttl] Wrote DOT -> 116.dot
[116.ttl] Wrote PNG -> 116.png
[118.t